In [1]:
import logging
import time
from pathlib import Path
from collections import deque
from typing import Dict, Any
import numpy as np

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import src.gymnasium_envs.convex_optimization_env

PROJECT_ROOT = Path().resolve().parent.parent

log_dir = PROJECT_ROOT / "logs"
model_dir = PROJECT_ROOT / "models"

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

2026-06-08 17:24:53 [INFO] Using device: cuda
2026-06-08 17:24:53 [INFO] GPU: NVIDIA GeForce GTX 1650
2026-06-08 17:24:53 [INFO] CUDA Version: 11.8


Adding status logging

In [3]:
class StatusCallback(BaseCallback):
    def __init__(self, window_size=1000):
        super().__init__()
        self.window = deque(maxlen=window_size)
        self.steps_to_converge = deque(maxlen=window_size)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            status = info.get("status")
            if status == "converged":
                self.window.append(1)
                self.steps_to_converge.append(info["iteration"])
            elif status == "diverged":
                self.window.append(-1)

        if len(self.window) > 0:
            self.logger.record("custom/converge_rate",
                self.window.count(1) / len(self.window))
            self.logger.record("custom/diverge_rate",
                self.window.count(-1) / len(self.window))

        if len(self.steps_to_converge) > 0:
            self.logger.record("custom/mean_steps_to_converge",
                np.mean(self.steps_to_converge))  # чем меньше тем лучше

        return True

class BestConvergeCallback(BaseCallback):
    def __init__(self, vec_normalize_env, model_save_path, vec_normalize_save_path, verbose=1):
        super().__init__(verbose)
        self.vec_normalize_env = vec_normalize_env
        self.model_save_path = model_save_path
        self.vec_normalize_save_path = vec_normalize_save_path
        self.best_converge_rate = -np.inf

    def _on_step(self) -> bool:
        converge_rate = self.logger.name_to_value.get("custom/converge_rate", None)

        if converge_rate is not None and converge_rate > self.best_converge_rate:
            self.best_converge_rate = converge_rate
            self.model.save(self.model_save_path)
            self.vec_normalize_env.save(self.vec_normalize_save_path)
            if self.verbose:
                logger.info(f"New best converge_rate: {converge_rate:.4f}")
        return True

Let's create a function for training

In [4]:
def train_model(config: Dict[str, Any], log_dir: str = "logs", model_dir: str = "models", add_time_penalty : bool = False, add_noise : bool = False, name_prefix = ""):
    in_features_name = "any" if config["in_features"] is None else config["in_features"]

    in_features_name = name_prefix + str(in_features_name)

    logger.info(f"Starting training for {in_features_name}D task. Timesteps: {config['timesteps']}, Envs: {config['n_envs']}")

    start_time    = time.time()
    model_path    = Path(f"{model_dir}/{in_features_name}d_convex")
    best_model_path = Path(f"{model_dir}/{in_features_name}d_convex_best")
    stats_path    = Path(f"{model_dir}/{in_features_name}d_convex_vec_normalize.pkl")

    model_path.parent.mkdir(parents=True, exist_ok=True)
    best_model_path.mkdir(parents=True, exist_ok=True)

    vec_env = None
    try:
        vec_env = make_vec_env(
            "convex_optimization_env/ConvexOptimization-v1",
            n_envs=config["n_envs"],
            env_kwargs={"in_features": config["in_features"],
                        "add_time_penalty" : add_time_penalty,
                        "add_noise" : add_noise}
        )
        vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True)

        best_converge_callback = BestConvergeCallback(
            vec_normalize_env=vec_env,
            model_save_path=str(best_model_path / "best_model"),
            vec_normalize_save_path=str(best_model_path / "best_vec_normalize.pkl"),
            verbose=1
        )

        model = PPO(
            "MultiInputPolicy",
            vec_env,
            verbose=1,
            tensorboard_log=f"{log_dir}/{in_features_name}d/",
            n_steps=config["n_steps"],
            n_epochs=config["n_epochs"],
            batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            gamma=config["gamma"],
            clip_range=config["clip_range"],
            ent_coef=config["ent_coef"],
            policy_kwargs=config["policy_kwargs"]
        )

        model.learn(
            total_timesteps=config["timesteps"],
            tb_log_name=f"convex_{in_features_name}d",
            callback=CallbackList([
                StatusCallback(),
                best_converge_callback
            ])
        )

        # Сохраняем финальную модель отдельно
        model.save(str(model_path))
        vec_env.save(str(stats_path))

        elapsed_time = (time.time() - start_time) / 60
        logger.info(f"Training for {in_features_name}D completed. Duration: {elapsed_time:.2f} min.")
        logger.info(f"Final model saved to: {model_path}")
        logger.info(f"Best model saved to: {best_model_path / 'best_model'}")

    except Exception as e:
        logger.error(f"Error during training for {in_features_name}D: {str(e)}", exc_info=True)
    finally:
        if vec_env is not None:
            vec_env.close()

In [5]:
config_base = {
    "in_features"   : None,
    "timesteps"     : 5_000_000, 
    "n_envs"        : 32,
    "n_steps"       : 2048,
    "n_epochs"      : 10, 
    "batch_size"    : 256, 
    "learning_rate" : 3e-4,
    "gamma"         : 0.99,
    "clip_range"    : 0.2,
    "ent_coef"      : 0.01,
    "policy_kwargs" : {
        "net_arch": dict(pi=[64, 64], vf=[64, 64])
    }
}

In [ ]:
#2D Convex

config = config_base
config["in_features"] = 2

train_model(config, log_dir, model_dir)

In [ ]:
#5D Convex

config = config_base
config["in_features"] = 5

train_model(config, log_dir, model_dir)

In [ ]:
#10D Convex

config = config_base
config["in_features"] = 10

train_model(config, log_dir, model_dir)

In [ ]:
#100D Convex

config = config_base
config["in_features"] = 100

train_model(config, log_dir, model_dir)

In [ ]:
#AnyD Convex

config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir)

In [ ]:
#AnyD Convex Time Penalty

config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir, add_time_penalty = True, name_prefix="time_penalty_")

In [6]:
#2D Convex Noise

config = config_base
config["in_features"] = 2

train_model(config, log_dir, model_dir, add_noise=True, name_prefix="noise_")

2026-06-08 17:25:02 [INFO] Starting training for noise_2D task. Timesteps: 5000000, Envs: 32
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/noise_2d/convex_noise_2d_3


2026-06-08 17:25:04 [INFO] New best converge_rate: 0.0000
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\numpy\linalg\_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:87: RuntimeWarning: overflow encountered in matmul
  return float(x.T @ self._A @ x + self._b @ x + self._c)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\gymnasium_envs\convex_optimization_env\envs\convex_optimization_v1.py:351: RuntimeWarning: overflow encountered in dot
  cos_sim = np.dot(grad_unit, prev_grad_unit)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:92: RuntimeWarning: overflow encountered in matmul
  return 2 * self._A @ x + self._b
C:\Users\Lolik\Documents\GitHub\Reinfor

---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 621      |
|    ep_rew_mean     | -221     |
| time/              |          |
|    fps             | 5979     |
|    iterations      | 1        |
|    time_elapsed    | 10       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 662          |
|    ep_rew_mean          | -238         |
| time/                   |              |
|    fps                  | 3192         |
|    iterations           | 2            |
|    time_elapsed         | 41           |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-06-08 18:30:13 [INFO] Training for noise_2D completed. Duration: 65.18 min.
2026-06-08 18:30:13 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_2d_convex
2026-06-08 18:30:13 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_2d_convex_best\best_model


In [7]:
#5D Convex Noise

config = config_base
config["in_features"] = 5

train_model(config, log_dir, model_dir, add_noise=True, name_prefix="noise_")

2026-06-08 18:31:01 [INFO] Starting training for noise_5D task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/noise_5d/convex_noise_5d_1


c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(
2026-06-08 18:31:01 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 572      |
|    ep_rew_mean     | -203     |
| time/              |          |
|    fps             | 1415     |
|    iterations      | 1        |
|    time_elapsed    | 46       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 562          |
|    ep_rew_mean          | -192         |
| time/                   |              |
|    fps                  | 931          |
|    iterations           | 2            |
|    time_elapsed         | 140          |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-06-08 19:17:09 [INFO] Training for noise_5D completed. Duration: 46.14 min.
2026-06-08 19:17:09 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_5d_convex
2026-06-08 19:17:09 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_5d_convex_best\best_model


In [8]:
#10D Convex Noise

config = config_base
config["in_features"] = 10

train_model(config, log_dir, model_dir, add_noise=True, name_prefix="noise_")

2026-06-08 19:17:09 [INFO] Starting training for noise_10D task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/noise_10d/convex_noise_10d_1


2026-06-08 19:17:09 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 631      |
|    ep_rew_mean     | -224     |
| time/              |          |
|    fps             | 4964     |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    converge_rate        | 0           |
|    diverge_rate         | 1           |
| rollout/                |             |
|    ep_len_mean          | 612         |
|    ep_rew_mean          | -208        |
| time/                   |             |
|    fps                  | 2811        |
|    iterations           | 2           |
|    time_elapsed         | 46          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_

2026-06-08 20:01:07 [INFO] Training for noise_10D completed. Duration: 43.96 min.
2026-06-08 20:01:07 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_10d_convex
2026-06-08 20:01:07 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_10d_convex_best\best_model


In [9]:
#100D Convex Noise

config = config_base
config["in_features"] = 100

train_model(config, log_dir, model_dir, add_noise=True, name_prefix="noise_")

2026-06-08 20:01:07 [INFO] Starting training for noise_100D task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/noise_100d/convex_noise_100d_1


2026-06-08 20:01:07 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 642      |
|    ep_rew_mean     | -232     |
| time/              |          |
|    fps             | 3963     |
|    iterations      | 1        |
|    time_elapsed    | 16       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 652          |
|    ep_rew_mean          | -227         |
| time/                   |              |
|    fps                  | 2364         |
|    iterations           | 2            |
|    time_elapsed         | 55           |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-06-08 20:47:39 [INFO] Training for noise_100D completed. Duration: 46.53 min.
2026-06-08 20:47:39 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_100d_convex
2026-06-08 20:47:39 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\noise_100d_convex_best\best_model
